# 1. Data Loading
---
This notebook is intended to be run both locally and on kaggle. Therefore, it needs accomodates the different data sources when loading the csvs. Before handling the data, imports will all be coalesced into a specified subsection.



## 1.1 Imports


In [5]:
import os

import pandas as pd


## 1.2 Data Loading

In [6]:
# 1. Automatically detect the environment
IS_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IS_KAGGLE:
    # Settings optimized for Kaggle's NVIDIA GPUs
    TREE_METHOD = "hist"  # XGBoost
    DEVICE_TYPE = "gpu"  # LightGBM (Kaggle uses OpenCL 'gpu', not 'cuda')
    CAT_TASK_TYPE = "GPU"  # CatBoost
    print("Environment: Kaggle (NVIDIA GPU Enabled)")
else:
    # Settings optimized for the local environment
    TREE_METHOD = "hist"  # XGBoost (Uses Metal/MPS natively via hist)
    DEVICE_TYPE = "cpu"  # LightGBM (Standard Mac wheels lack GPU support)
    CAT_TASK_TYPE = "CPU"  # CatBoost (Mac GPU support is experimental; CPU is faster)
    print("Environment: Local Mac (CPU fallback for LightGBM/CatBoost)")

Environment: Local Mac (CPU fallback for LightGBM/CatBoost)


In [7]:
def load_data():
    """
    try to load the data from the local directory, if not found, load it from the Kaggle dataset.
    Output the three dataframes: train, test, and submission.
    """
    train_file_dir_name = "/train.csv"
    sample_submission_file_dir_name = "/sample_submission.csv"
    test_file_dir_name = "/test.csv"

    if IS_KAGGLE:
        # check if the data files are in kaggle direcotry
        # /kaggle/input/competitions/playground-series-s6e9
        kaggle_input_dir = "/kaggle/input/competitions/"
        competition_name = os.listdir(kaggle_input_dir)

        print("Trying to load data from kaggle input directory...")
        data_dir = "/kaggle/input/competitions/playground-series-s6e9/"
        train = pd.read_csv(
            kaggle_input_dir + competition_name[-1] + train_file_dir_name
        )
        test = pd.read_csv(kaggle_input_dir + competition_name[-1] + test_file_dir_name)
        submission = pd.read_csv(
            kaggle_input_dir + competition_name[-1] + sample_submission_file_dir_name
        )
        print("Data loaded successfully from kaggle input directory.")
    else:
        print("Trying to load data from local directory...")
        # Load data from local directory
        data_dir = os.getcwd() + "/data"
        train = pd.read_csv(data_dir + train_file_dir_name)
        test = pd.read_csv(data_dir + test_file_dir_name)
        submission = pd.read_csv(data_dir + sample_submission_file_dir_name)
        print("Data loaded successfully from local directory.")

    return train, test, submission

In [8]:
train_df, test_df, submission_df = load_data()

Trying to load data from local directory...
Data loaded successfully from local directory.


# 2. Data exploration
---
Having organised the data into dataframes that can be explored and prepared for preprocessing, the necessary quality checks can be done. In this section, we will explore the data for missing values, extreme values and mislabelled data types. 

## 2.1 Check for missing values/null values

In [9]:
def check_missing_values(df):
    """
    Check for missing values in the dataframe and print the count of missing values for each column.
    """
    temp_df = df.copy()
    # check if the dataframe is empty
    if temp_df.empty:
        print("The dataframe is empty. No missing values to check.")
        return
    # check that the dataframe has missing values
    if temp_df.isnull().sum().sum() == 0:
        print("The dataframe has no missing values.")
        return
    # Iterate through each column and check the number of missing values
    for column in temp_df.columns:
        missing_count = temp_df[column].isnull().sum()
        if missing_count > 0:
            # print column name
            # number of missing values againts total number of rows in the column
            total_rows = temp_df.shape[0]
            print(
                f"{column:<25}"
                f"Missing Values: {missing_count:<10}"
                f"Total Rows: {total_rows:<10}"
                f"Percentage Missing: {missing_count / total_rows * 100:.2f}%"
            )

In [10]:
check_missing_values(train_df)


The dataframe has no missing values.


In [11]:
check_missing_values(test_df)

The dataframe has no missing values.
